In [107]:
pip install pandas plotnine requests Leaflet asyncio noaa-cdo-api geopy arcgis


Defaulting to user installation because normal site-packages is not writeable
  Using cached keyring-25.7.0-py3-none-any.whl.metadata (21 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached pywin32_ctypes-0.2.3-py3-none-any.whl.metadata (3.9 kB)
  Using cached jaraco.classes-3.4.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached jaraco_context-6.1.2-py3-none-any.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/13.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.3 MB ? eta -:--:--
    --------------------------------------- 0.3/13.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.3 MB 909.6 kB/s eta 0:00:15
   -- ------------------------------------- 0.8/13.3 MB 890.7 kB/s eta 0:00:15
   --- ------------------------------------ 1.0/13.3 MB 983.0 kB/s eta 0:00:13
   --- ------------------------------------ 1.0/1

In [1]:
#Found from https://stackoverflow.com/questions/18828890/how-to-use-the-noaa-api-to-query-past-weather-data-for-a-given-set-of-coordinate
#Code from https://github.com/crvaden/NOAA_API_v2/blob/Update/noaa_api_v2.py
#This is apparently a complete reasonable way to re-use code. As Dr. Letche says, don't reinvent the wheel.
# NOAA API V2
# Documentation can be found at
# http://www.ncdc.noaa.gov/cdo-web/webservices/v2
import requests


class NOAAData(object):
    def __init__(self, token):
        # NOAA API Endpoint
        self.url = 'https://www.ncdc.noaa.gov/cdo-web/api/v2/'
        self.h = dict(token=token)

    def poll_api(self, req_type, payload):
        # Initiate http request - kwargs are constructed into a dict and passed as optional parameters
        # Ex (limit=100, sortorder='desc', startdate='1970-10-03', etc)
        r = requests.get(self.url + req_type, headers=self.h, params=payload)

        if r.status_code != 200:  # Handle erroneous requests
            print("Error: " + str(r.status_code))
        else:
            r = r.json()
            try:
                return r['results']  # Most JSON results are nested under 'results' key
            except KeyError:
                return r  # for non-nested results, return the entire JSON string

    # Fetch available datasets
    # http://www.ncdc.noaa.gov/cdo-web/webservices/v2#datasets
    def datasets(self, **kwargs):
        req_type = 'datasets'
        return self.poll_api(req_type, kwargs)

    # Fetch data categories
    # http://www.ncdc.noaa.gov/cdo-web/webservices/v2#dataCategories
    def data_categories(self, **kwargs):
        req_type = 'datacategories'
        return self.poll_api(req_type, kwargs)

    # Fetch data types
    # http://www.ncdc.noaa.gov/cdo-web/webservices/v2#dataTypes
    def data_types(self, **kwargs):
        req_type = 'datatypes'
        return self.poll_api(req_type, kwargs)

    # Fetch available location categories
    # http://www.ncdc.noaa.gov/cdo-web/webservices/v2#locationCategories
    def location_categories(self, **kwargs):
        req_type = 'locationcategories'
        return self.poll_api(req_type, kwargs)

    # Fetch all available locations
    # http://www.ncdc.noaa.gov/cdo-web/webservices/v2#locations
    def locations(self, **kwargs):
        req_type = 'locations'
        return self.poll_api(req_type, kwargs)

    # Fetch All available stations
    # http://www.ncdc.noaa.gov/cdo-web/webservices/v2#stations
    def stations(self, h, p, **kwargs):
        req_type = 'stations'
        return self.poll_api(req_type, kwargs)

    # Fetch information about specific dataset
    def dataset_spec(self, set_code, **kwargs):
        req_type = 'datacategories/' + set_code
        return self.poll_api(req_type, kwargs)

    # Fetch data
    # http://www.ncdc.noaa.gov/cdo-web/webservices/v2#data
    def fetch_data(self, **kwargs):
        req_type = 'data'
        return self.poll_api(req_type, kwargs)

In [2]:
#Import required libraries
import pandas as pd
import plotnine as p9
import numpy as np
import requests
import leaflet
import asyncio
from noaa_cdo_api import NOAAClient, Extent
import time

In [3]:
def glimpse(df):
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    for col in df.columns:
        print(f"$ {col} <{df[col].dtype}> {df[col].head().values}")

In [ ]:
token = ""
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/"
df_new = pd.read_csv("/Users/jwsqu/Desktop/Bio456_Project/Journey North hummingbird data 2021-2026.csv")
df_old = pd.read_csv("/Users/jwsqu/Desktop/Bio456_Project/UWMadisonArb_JNorth_Project_Hummingbirds_1996-2020.csv")

C:\Users\jwsqu\AppData\Local\Temp\ipykernel_14864\2017043100.py:4: DtypeWarning: Columns (8: school, 10: image_url) have mixed types. Specify dtype option on import or set low_memory=False.


In [ ]:
#Now I need to join the two data sets. 
glimpse(df_new)
df_new_clean = df_new[['Latitude','Longitude','Sighting Date']]


Rows: 26996
Columns: 25
$ ID <int64> [1526499327 1526499338 1526499339 1526499394 1526499559]
$ Project <str> <StringArray>
[nan, nan, nan, nan, nan]
Length: 5, dtype: str
$ Species <str> <StringArray>
[nan, nan, nan, nan, nan]
Length: 5, dtype: str
$ Reporting Category <str> <StringArray>
['Ruby-throated Hummingbird (All Other Observations)',
 'Ruby-throated Hummingbird (All Other Observations)',
 'Ruby-throated Hummingbird (All Other Observations)',
 'Ruby-throated Hummingbird (All Other Observations)',
 'Ruby-throated Hummingbird (All Other Observations)']
Length: 5, dtype: str
$ Life Stage <str> <StringArray>
[nan, nan, nan, nan, nan]
Length: 5, dtype: str
$ Number <float64> [1. 2. 2. 2. 1.]
$ Number Unknown <float64> [nan nan nan nan nan]
$ Sex <str> <StringArray>
[nan, nan, nan, nan, nan]
Length: 5, dtype: str
$ First Observed <float64> [nan nan nan nan nan]
$ Customer ID <float64> [10035290. 10036042. 10032753. 10035290. 10036372.]
$ Nearest <str> <StringArray>
['Lake Wales', 'M

In [7]:
glimpse(df_old)
df_old_clean = df_old[['latitude','longitude','sighting_date']]

Rows: 164841
Columns: 16
$ id <int64> [856986234 858828649 860774855 861039959 861459012]
$ sighting_date <str> <StringArray>
['1997-02-26', '1997-03-13', '1997-04-03', '1997-04-12', '1997-04-19']
Length: 5, dtype: str
$ species <str> <StringArray>
['Hummingbird, Ruby-throated (FIRST)', 'Hummingbird, Ruby-throated (FIRST)',
 'Hummingbird, Ruby-throated (FIRST)', 'Hummingbird, Ruby-throated (FIRST)',
 'Hummingbird, Ruby-throated (FIRST)']
Length: 5, dtype: str
$ number <int64> [1 1 1 1 1]
$ latitude <float64> [38.596 29.959 28.14  35.916 41.237]
$ longitude <float64> [-122.469  -93.918  -82.712  -85.795  -85.854]
$ flag_location <int64> [0 0 0 0 0]
$ comments <str> <StringArray>
['although we don't have the ruby-throated her in California, we just want to report that on Friday the 21st, the first Anna's was sighted at a feeding station.  It has been seen feeding from shrub blossoms and the feeder every day since Friday.  Also a rufous was seen on Saturday the 22nd.  The are a coastal sp

In [ ]:
df_new_clean = df_new_clean.rename(columns={'Latitude':'latitude','Longitude':'longitude','Sighting Date':'sighting_date'})
df_full = pd.concat([df_old_clean, df_new_clean])
df_full['date_scraped'] = False

In [82]:
glimpse(df_full)
df_full['zipcode'] = 'Empty'

Rows: 191837
Columns: 5
$ latitude <float64> [38.596 29.959 28.14  35.916 41.237]
$ longitude <float64> [-122.469  -93.918  -82.712  -85.795  -85.854]
$ sighting_date <str> <StringArray>
['1997-02-26', '1997-03-13', '1997-04-03', '1997-04-12', '1997-04-19']
Length: 5, dtype: str
$ date_scraped <bool> [False False False False False]
$ zipcode <object> [nan nan nan nan nan]


SyntaxError: invalid syntax (2946069985.py, line 13)

In [ ]:
#df_full = df.reset_index(drop=False).rename(columns={"index": "row_id"})

In [ ]:
#df_full.to_csv("unzipped_data.csv")

In [ ]:
#df_full.to_csv("hummingbird_zip.csv")

In [ ]:
#The empty strings below take a GIS username and password. It works by keeping a CSV and appending the results of the scrape onto it.
#It keeps track of the index number, and uses that to avoid scraping the same thing twice. It technically doesn't need a rate limit, 
#but without it it crashes my python install. The rest basically just sends a point to ArcGIS servers, and returns the zipcode its 
#located in, and makes a new line, appends it, and then repeats indefinately. I didn't bother making it stop itself, so if you run 
#it it would likely just keep executing nothing. It worked about twice as fast as another method I tried, which was to do the same,
#but with a free package "geopy". the rate limit began to be a problem, however, and I learned that I could use arcgis's package
#from some forums on stackoverflow. 
from arcgis.gis import GIS
from arcgis.geocoding import reverse_geocode
from arcgis.geometry import Point
import os

spatial = {"wkid": 4326}
gis = GIS("https://www.arcgis.com", "", "")

output_file = "hummingbird_zip.csv"
checkpoint_col = "row_id"

if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    processed_ids = set(processed_df[checkpoint_col])
    print(f"Resuming: {len(processed_ids)} already processed.")
else:
    processed_df = pd.DataFrame()
    processed_ids = set()

df = pd.read_csv("unzipped_data.csv")

df_to_process = df[~df[checkpoint_col].isin(processed_ids)]

for index, row in df_to_process.iterrows():
    time.sleep(.5)
    lat = row['latitude']
    lon = row['longitude']
    row_id = row[checkpoint_col]
    
    zip_code = None
    try:
        location = Point({"x": lon, "y": lat, "spatialReference": spatial})
        results = reverse_geocode(location = location, feature_types="Postal", out_sr=4326)
        if results:
            attrs = results.get("attributes", {})
            zip_code = attrs.get("Postal") or attrs.get("ZIP") or attrs.get("ZIP4")
            
            if not zip_code:
                address = results.get("address", {})
                zip_code = address.get("Postal") or address.get("ZIP")
    except Exception as e:
        print(f"Error on row {row_id}: {e}")
        zip_code = None
    result_row = pd.DataFrame([{checkpoint_col: row_id,
                                'latitude': lat,
                                'longitude': lon,
                                'zip_code': zip_code
                                }])
    result_row.to_csv(output_file, mode='a', header=not os.path.exists(output_file), index = False)

Resuming: 20972 already processed.
Error on row 21209: Cannot perform query. Invalid query parameters.
Unable to find address for the specified location.
(Error Code: 400)
Error on row 21672: Cannot perform query. Invalid query parameters.
Unable to find address for the specified location.
(Error Code: 400)
Error on row 21729: Cannot perform query. Invalid query parameters.
Unable to find address for the specified location.
(Error Code: 400)
Error on row 22084: Cannot perform query. Invalid query parameters.
Unable to find address for the specified location.
(Error Code: 400)
Error on row 23318: Cannot perform query. Invalid query parameters.
Unable to find address for the specified location.
(Error Code: 400)
Error on row 23615: Cannot perform query. Invalid query parameters.
Unable to find address for the specified location.
(Error Code: 400)
Error on row 23848: Cannot perform query. Invalid query parameters.
Unable to find address for the specified location.
(Error Code: 400)
Error 

KeyboardInterrupt: 

In [ ]:
df.to_csv('df_full.csv')

In [4]:
#basic usage of the class
from noaa_api_v2 import NOAAData
data = NOAAData(token)

categories = data.data_categories(locationid='FIPS:37', sortfield='name')

for i in categories:
    print(i)


{'name': 'Air Temperature', 'id': 'TEMP'}
{'name': 'Annual Agricultural', 'id': 'ANNAGR'}
{'name': 'Annual Degree Days', 'id': 'ANNDD'}
{'name': 'Annual Precipitation', 'id': 'ANNPRCP'}
{'name': 'Annual Temperature', 'id': 'ANNTEMP'}
{'name': 'Autumn Agricultural', 'id': 'AUAGR'}
{'name': 'Autumn Degree Days', 'id': 'AUDD'}
{'name': 'Autumn Precipitation', 'id': 'AUPRCP'}
{'name': 'Autumn Temperature', 'id': 'AUTEMP'}
{'name': 'Computed', 'id': 'COMP'}
{'name': 'Computed Agricultural', 'id': 'COMPAGR'}
{'name': 'Degree Days', 'id': 'DD'}
{'name': 'Dual-Pol Moments', 'id': 'DUALPOLMOMENT'}
{'name': 'Echo Tops', 'id': 'ECHOTOP'}
{'name': 'Evaporation', 'id': 'EVAP'}
{'name': 'Hydrometeor Type', 'id': 'HYDROMETEOR'}
{'name': 'Land', 'id': 'LAND'}
{'name': 'Miscellany', 'id': 'MISC'}
{'name': 'Other', 'id': 'OTHER'}
{'name': 'Overlay', 'id': 'OVERLAY'}
{'name': 'Precipitation', 'id': 'PRCP'}
{'name': 'Pressure', 'id': 'PRES'}
{'name': 'Reflectivity', 'id': 'REFLECTIVITY'}
{'name': 'Sky cov

It works, so I can go ahead with further testing, viewing it in a pandas table


In [5]:
datasets = data.datasets()
for i in datasets:
    print(i)

{'uid': 'gov.noaa.ncdc:C00861', 'mindate': '1763-01-01', 'maxdate': '2026-09-26', 'name': 'Daily Summaries', 'datacoverage': 1, 'id': 'GHCND'}
{'uid': 'gov.noaa.ncdc:C00946', 'mindate': '1763-01-01', 'maxdate': '2026-09-01', 'name': 'Global Summary of the Month', 'datacoverage': 1, 'id': 'GSOM'}
{'uid': 'gov.noaa.ncdc:C00947', 'mindate': '1763-01-01', 'maxdate': '2026-01-01', 'name': 'Global Summary of the Year', 'datacoverage': 1, 'id': 'GSOY'}
{'uid': 'gov.noaa.ncdc:C00345', 'mindate': '1991-06-05', 'maxdate': '2026-09-28', 'name': 'Weather Radar (Level II)', 'datacoverage': 0.95, 'id': 'NEXRAD2'}
{'uid': 'gov.noaa.ncdc:C00708', 'mindate': '1994-05-20', 'maxdate': '2026-09-24', 'name': 'Weather Radar (Level III)', 'datacoverage': 0.95, 'id': 'NEXRAD3'}
{'uid': 'gov.noaa.ncdc:C00821', 'mindate': '2010-01-01', 'maxdate': '2010-01-01', 'name': 'Normals Annual/Seasonal', 'datacoverage': 1, 'id': 'NORMAL_ANN'}
{'uid': 'gov.noaa.ncdc:C00823', 'mindate': '2010-01-01', 'maxdate': '2010-12-31

In [ ]:
#This was an attempt to get the scraping script above to work for NOAA Data. 
#I realized this would take way too long, as I would have to use it 
# seperately for each variable. this would have taken about a month,
# but I was able to find a downloadable form of the data.
# Currently i have them stored in Weather_Data.7zip
# I have found another possibly better source at https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/
"""output_file = "hummingbird_weather.csv"
checkpoint_col = "row_id"

data = NOAAData(token)

if os.path.exists(output_file):
    processed_df_weather = pd.read_csv(output_file)
    processed_ids_weather = set(processed_df[checkpoint_col])
    print(f"Resuming: {len(processed_ids)} already processed.")
else:
    processed_df_weather = pd.DataFrame()
    processed_ids_weather = set()

df = pd.read_csv("hummingbird_zip.csv")

df_to_process_weather = df[~df[checkpoint_col].isin(processed_ids)]

for index, row in df_to_process_weather.iterrows():
    time.sleep(.5)
    zip =row['zip_code']
    date = row['sighting_date']
    
    
    try:
        
        results = NOAAClient.get_data()
        if results:
            attrs = results.get("attributes", {})
            zip_code = attrs.get("Postal") or attrs.get("ZIP") or attrs.get("ZIP4")
            
            if not zip_code:
                address = results.get("address", {})
                zip_code = address.get("Postal") or address.get("ZIP")
    except Exception as e:
        print(f"Error on row {row_id}: {e}")
        zip_code = None
    result_row = pd.DataFrame([{checkpoint_col: row_id,
                                'latitude': lat,
                                'longitude': lon,
                                'zip_code': zip_code
                                }])
    result_row.to_csv(output_file, mode='a', header=not os.path.exists(output_file), index = False)"""
            
